<a href="https://colab.research.google.com/github/Isabela-Tellez/BootcampIA/blob/main/07.%20Julio-13/Optimizaci%C3%B3n_Multi_M%C3%A9trica_RandomizedSearchCV_(ROC_AUC_%26_F1_Score).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Reto Práctico: Detección de Fraude en Tarjetas de Crédito
Define los objetivos, el contexto de negocio y los desafíos analíticos a resolver antes de iniciar la actividad:

*   📊 Contexto del Problema: Optimización de un modelo predictivo con el dataset Credit Card Fraud Detection (ULB), el cual presenta un desbalanceo extremo con solo un 2% de transacciones fraudulentas.

*   🔮 Optimización Multi-Métrica: Configuración avanzada de RandomizedSearchCV para evaluar en paralelo las métricas ROC-AUC y F1-Score, analizando el uso de refit para activar el método .predict() y la lectura de cv_results_.

*   ⚖️ Métricas y Umbrales: Justificación de por qué el Accuracy es inválido ante un desbalanceo del 2% y análisis de cómo el umbral (threshold) altera el balance entre Precision y Recall.

*   🧠 Restricción Ética: Evaluación del impacto real en el usuario al priorizar el Recall sobre la Precision, traduciendo el trade-off matemático a una estrategia de comunicación para equipos no técnicos.

## 💳 Preparación y Limpieza del Dataset de Fraude
Realiza la etapa esencial de preprocesamiento de datos antes de entrenar el modelo de Machine Learning:  

*   📥 Carga de Datos: Usa pandas para leer el archivo creditcard.csv con el histórico de transacciones.

*   🧼 Limpieza de Nulos: Remueve filas vacías en la columna objetivo Class para evitar fallas matemáticas.

*   ✂️ Separación: Aísla las características descriptivas (X) de la etiqueta de fraude (y).

*   ⚖️ División Estratificada: Separa un 80% para entrenamiento y 20% para pruebas (test_size=0.2), manteniendo con stratify=y la proporción original de fraudes para evitar sesgos.




In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Carga del archivo CSV del reto
df = pd.read_csv("creditcard.csv")

# Eliminación de cualquier fila donde la columna 'Class' tenga un valor nulo
df = df.dropna(subset=['Class'])

# Separación de las características (X) de la etiqueta objetivo (y)
X = df.drop(columns=['Class']) # 'Class' es la columna objetivo del reto
y = df['Class']

# Divición de entrenamiento y prueba (guardando el 20% para test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 🛡️ Configuración de la Validación Cruzada Estratificada
Establece la estrategia para evaluar el modelo de forma robusta frente a datos altamente desbalanceados:

*   📦 Validador Especializado: Carga StratifiedKFold para mantener la proporción exacta de las clases en cada partición del dataset.

*   ✂️ Divisiones (n_splits=5): Configura 5 bloques independientes para entrenar y validar el modelo 5 veces de forma rotativa.

*   🎲 Mezclado (shuffle=True): Aleatoriza las transacciones antes de los cortes para evitar sesgos por el orden de registro original.

*   🌱 Reproducibilidad (random_state=42): Fija una semilla para garantizar que las divisiones sean idénticas en cada ejecución.

*   ⚖️ Control de Desbalance: Asegura que el porcentaje crítico de fraudes esté representado proporcionalmente en cada pliegue.

In [13]:
from sklearn.model_selection import StratifiedKFold

# Configuración de 5 divisiones (splits), activación del mezclado (shuffle)
# y uso de semilla (random_state) para que siempre sea el mismo resultado.
cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 🎯 Configuración del Modelo y Espacio de Búsqueda Probabilístico
Define el algoritmo y los rangos dinámicos de hiperparámetros que se explorarán de forma inteligente:

*   🧬 Muestreo Estadístico: Carga randint y uniform de scipy.stats para buscar de forma continua o discreta en lugar de usar listas fijas.

*   📊 Rangos de Optimización (param_dist): Delimita el espacio de búsqueda del bosque:

    *   n_estimators: Explora de 50 a 300 árboles para determinar el tamaño ideal del modelo.

    *   max_depth: Evalúa profundidades de 3 a 20 niveles para prevenir el sobreajuste (overfitting).

    *   min_samples_split: Muestrea porcentajes del 1% al 20% para regular la división de los nodos.

*   🌲 Modelo Base: Inicializa el clasificador RandomForestClassifier para procesar el conjunto de transacciones.

*   🌱 Réplica Exacta (random_state=42): Fija una semilla numérica para garantizar la reproducibilidad y aislar el impacto real de los hiperparámetros.

In [14]:
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier

# Definición de los rangos de búsqueda como distribuciones
param_dist = {
    "n_estimators": randint(50, 300),          # Explora enteros entre 50 y 300
    "max_depth": randint(3, 20),               # Explora profundidades de árbol de 3 a 20
    "min_samples_split": uniform(0.01, 0.2)    # Explora porcentajes decimales continuos
}

# Deja definido tu modelo base con una semilla para evitar aleatoriedad
modelo = RandomForestClassifier(random_state=42)

## ⚡ Configuración de Búsqueda Multi-Métrica y Optimización con RandomizedSearchCV
Este bloque ejecuta la búsqueda automática e inteligente de la mejor configuración para el modelo:

*   🧠 Configuración Integral: Une el clasificador, el espacio de parámetros probabilísticos y la validación cruzada estratificada.

*   🎲 Muestreo Veloz (n_iter=5): Evalúa solo 5 combinaciones al azar, reduciendo drásticamente el tiempo frente a un análisis exhaustivo.

*   📊 Doble Evaluación: Mide en paralelo la discriminación (ROC-AUC) y el balance precisión/sensibilidad (F1-Score).

*   ⚖️ Decisión Automática (refit='f1'): Selecciona el mejor modelo según el F1-Score y lo deja listo para usar .predict().

*   🚀 Cómputo en Paralelo (n_jobs=-1): Utiliza todos los procesadores de Colab para acelerar el entrenamiento simultáneo.

*   ⏱️ Muestreo Controlado: Ajusta el modelo (.fit) con las primeras 20,000 filas para evitar bloqueos por memoria y agilizar la entrega de resultados.

In [16]:
from sklearn.model_selection import RandomizedSearchCV

# Configuración del buscador con todo lo construido
busqueda = RandomizedSearchCV(
    estimator=modelo,
    param_distributions=param_dist,
    n_iter=5,                       # Probará 5 combinaciones aleatorias
    scoring=['roc_auc', 'f1'],       # Evaluación de ambas métricas a la vez
    refit='f1',                      # Elige 'f1' o 'roc_auc' como el criterio ganador
    cv=cv_estrategia,                # Estrategia del Paso 1
    random_state=42,
    n_jobs=-1                        # Usa todos los procesadores para ir más rápido
)

# ¡Hora de entrenar! Ajusta el modelo con tus datos del reto
#busqueda.fit(X_train, y_train)
busqueda.fit(X_train[:20000], y_train[:20000])

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=RandomForestClassifier(random_state=42), n_iter=5,
                   n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b2111b1e540>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7b2111b21790>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b2111b8ae10>},
                   random_state=42, refit='f1', scoring=['roc_auc', 'f1'])

## 📊 Extracción e Interpretación de Resultados Multi-Métrica
Convierte y ordena los datos técnicos de la optimización en una estructura legible para facilitar la toma de decisiones del negocio:

*   📈 Conversión a DataFrame: Transforma el diccionario estructurado busqueda.cv_results_ en una tabla de Pandas (tabla_resultados) para realizar análisis de datos de forma ágil.

*   🎯 Filtrado de Métricas Clave: Aísla únicamente las columnas esenciales solicitadas por el equipo de riesgo: los hiperparámetros evaluados (params), el rendimiento del balance precisión/recall (mean_test_f1) y la capacidad de discriminación del modelo (mean_test_roc_auc).

*   🥇 Ordenamiento Estratégico: Clasifica las filas de mayor a menor (ascending=False) tomando como referencia el F1-Score (el criterio definido en el parámetro refit). Esto sitúa automáticamente la combinación ganadora de hiperparámetros en la primera fila de la tabla.

In [17]:
import pandas as pd

# Conversión a diccionario de resultados en una tabla de Pandas
tabla_resultados = pd.DataFrame(busqueda.cv_results_)

# Filtro de las columnas clave para comparar el rendimiento de cada combinación
columnas_interes = [
    'params',
    'mean_test_f1',
    'mean_test_roc_auc'
]

# Muestreo de la tabla ordenada de mejor a peor según el F1-Score
tabla_resultados[columnas_interes].sort_values(by='mean_test_f1', ascending=False)

,params,mean_test_f1,mean_test_roc_auc
1,"{'max_depth': 13, 'min_samples_split': 0.16593...",0.679274,0.960678
3,"{'max_depth': 13, 'min_samples_split': 0.10184...",0.679274,0.955865
2,"{'max_depth': 9, 'min_samples_split': 0.099166...",0.679274,0.954080
4,"{'max_depth': 6, 'min_samples_split': 0.038573...",0.679274,0.958908
0,"{'max_depth': 9, 'min_samples_split': 0.169308...",0.660226,0.959759


##🧠 Responsabilidad Ética y Estrategia de Negocio: Trade-off de Erreurs
Documenta la justificación analítica y el impacto humano detrás de la calibración final del modelo, resolviendo la restricción ética del reto:

*   👤 Impacto en Usuarios Reales: Al maximizar el Recall se eliminan los fraudes ocultos, protegiendo los fondos del cliente. El costo colateral de reducir la Precision es el aumento de falsas alarmas, traduciéndose en tarjetas legítimas bloqueadas por error en compras cotidianas.

*   🗣️ Comunicación No Técnica: Traduce el dilema matemático a métricas de negocio, comparando el alto costo financiero de asumir pérdidas por fraudes reales frente al costo operativo menor de gestionar alertas preventivas.

*   🛠️ Mitigación Operativa: Propone desplegar el modelo junto con una estrategia de experiencia de usuario (UX) fluida, utilizando notificaciones push o SMS de validación inmediata para desbloquear la transacción en segundos sin romper la confianza del cliente.